## ML_1026: Cross-Entropy Loss

**Difficulty**: Easy | **TorchCode**: #16

### Problem
Implement numerically stable cross-entropy loss from scratch. Use log-sum-exp trick: compute log probabilities by subtracting logsumexp, then index into the correct class.

### Formula
$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^N \log p_{y_i}, \quad \log p_i = x_i - \log\sum_j e^{x_j}$$

In [ ]:
import torch

# CE = -log(softmax(logits)[target_class])

def cross_entropy_loss(logits, targets):
    log_probs = logits - torch.logsumexp(logits, dim=-1, keepdim=True)
    return -log_probs[torch.arange(targets.shape[0]), targets].mean()

def cross_entropy_loss(logits, targets):
    targets = targets.unsqueeze(-1)
    max_value = torch.max(logits, dim=-1, keepdim = True).values
    logits = logits - max_value
    e_x = torch.exp(logits)
    e_x_sum = torch.sum(e_x, dim = -1, keepdim=True)
    softmax = e_x / e_x_sum
    
    targets_prob = torch.gather(softmax, dim = -1, index = targets)
    loss = -1 * torch.log(targets_prob)
    loss = torch.sum(loss)
    return loss / logits.shape[0]

In [ ]:
import torch

# --- Test 1: Matches F.cross_entropy ---
torch.manual_seed(0)
logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))
out = cross_entropy_loss(logits, targets)
ref = torch.nn.functional.cross_entropy(logits, targets)
assert torch.allclose(out, ref, atol=1e-5)

# --- Test 2: Numerical stability with large logits ---
logits = torch.tensor([[1000., 0., 0.], [0., 1000., 0.]])
targets = torch.tensor([0, 1])
out = cross_entropy_loss(logits, targets)
assert not torch.isnan(out) and not torch.isinf(out)
assert out.item() < 0.01

# --- Test 3: Scalar output ---
assert cross_entropy_loss(torch.randn(8, 5), torch.randint(0, 5, (8,))).dim() == 0

# --- Test 4: Gradient flow ---
logits = torch.randn(8, 5, requires_grad=True)
cross_entropy_loss(logits, torch.randint(0, 5, (8,))).backward()
assert logits.grad is not None

print('All tests passed!')